In [1]:
import cupy as cp
import cupyx

In [2]:
class Flatten:
    def __init__(self):
        self.x_shape = None

    def forward(self, x):
        self.x_shape = x.shape
        assert len(self.x_shape) > 1, f"Expects at least 2D input, got {len(self.x_shape)}"
        out = x.reshape(self.x_shape[0], -1)
        return out

    def backward(self, grad_output):
        assert self.x_shape is not None, "forward() has not been called"
        return grad_output.reshape(self.x_shape)

In [3]:
import cupy as cp

# --- forward ---
x = cp.random.randn(2, 3, 4, 5).astype(cp.float32)
layer = Flatten()

out = layer.forward(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)

assert out.shape == (2, 3 * 4 * 5)
print("Forward shape test passed.")


# --- backward ---
grad_output = cp.random.randn(*out.shape).astype(cp.float32)
grad_input = layer.backward(grad_output)

print("Grad output shape:", grad_output.shape)
print("Grad input shape:", grad_input.shape)

assert grad_input.shape == x.shape
print("Backward shape test passed.")

# --- value ---
x = cp.random.randn(2, 3, 4).astype(cp.float32)
layer = Flatten()

out = layer.forward(x)

assert cp.allclose(out, x.reshape(2, -1))
print("Forward value preservation test passed.")

grad_output = cp.random.randn(*out.shape).astype(cp.float32)
grad_input = layer.backward(grad_output)

assert cp.allclose(grad_input, grad_output.reshape(x.shape))
print("Backward value preservation test passed.")

Input shape: (2, 3, 4, 5)
Output shape: (2, 60)
Forward shape test passed.
Grad output shape: (2, 60)
Grad input shape: (2, 3, 4, 5)
Backward shape test passed.
Forward value preservation test passed.
Backward value preservation test passed.
